# Ovarian Cancer Data Analysis — Interactive Dashboard
### Intermediate-level machine learning project · No images · Google Colab

**Educational demonstration only.** All 6,000 records and labels are **synthetically generated**. The outcome is a **simulated pattern**, not verified ovarian cancer status. The dashboard must **never** be used for diagnosis, screening, or clinical decisions.

**What you get:** 6,000-row dataset • data exploration • 3 machine-learning models • model performance metrics • interactive risk-pattern explorer • batch CSV predictions • downloadable model and data.

**How to run:** In Colab, choose **Runtime → Run all**. The final cell starts a Gradio dashboard. To make a shareable Colab notebook URL, save a copy to Drive and click **Share**.

In [ ]:
# 1. Install simple dashboard dependencies (run in Colab)
%pip -q install "gradio>=4.44,<6" plotly scikit-learn pandas joblib seaborn

## 1 · Create the synthetic data
The dataset is created by the notebook itself, so no upload is needed. Values are fictitious and should not be interpreted as medical reference intervals.

In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import gradio as gr
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, RocCurveDisplay)

rng = np.random.default_rng(42)
n = 6000
age = rng.integers(20, 81, n)
menopause = np.where(age < 45, rng.binomial(1, .10, n),
                     np.where(age >= 55, rng.binomial(1, .85, n), rng.binomial(1, .55, n)))
family = rng.binomial(1, .14, n)
bloating = rng.binomial(1, .28, n)
pelvic = rng.binomial(1, .24, n)
latent = rng.binomial(1, .37, n)
ca125 = np.clip(rng.lognormal(np.log(18 + latent * 34), .55, n), 4, 650).round(1)
he4 = np.clip(rng.lognormal(np.log(46 + latent * 34 + menopause * 12), .37, n), 12, 450).round(1)
lesion = np.clip(rng.normal(2.8 + latent * 2.1, 1.25, n), .4, 12).round(1)

# Fictional mathematical rule used ONLY to create mock training labels
score = (-3.65 + .025 * (age - 40) + .65 * menopause + .55 * family
         + .60 * bloating + .60 * pelvic + .016 * (ca125 - 30)
         + .020 * (he4 - 50) + .40 * (lesion - 3))
probability = 1 / (1 + np.exp(-np.clip(score, -12, 12)))
labels = rng.binomial(1, probability)

data = pd.DataFrame({
    "Age": age, "CA125_U_mL": ca125, "HE4_pmol_L": he4,
    "Lesion_Size_cm": lesion, "Postmenopausal": menopause,
    "Family_History": family, "Persistent_Bloating": bloating,
    "Pelvic_Pain": pelvic, "Simulated_Pattern": labels
})
data.to_csv("ovarian_synthetic_6000.csv", index=False)
print("Rows:", len(data), "| Columns:", len(data.columns))
display(data.head())

## 2 · Explore dataset

In [ ]:
print("Outcome counts:")
print(data["Simulated_Pattern"].value_counts().rename(index={0:"Pattern 0",1:"Pattern 1"}))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data["Age"].hist(ax=axes[0], bins=18)
axes[0].set_title("Age distribution")
axes[0].set_xlabel("Age")
data["Simulated_Pattern"].value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set_title("Synthetic outcome balance")
axes[1].set_xticklabels(["Pattern 0", "Pattern 1"], rotation=0)
plt.tight_layout()
plt.show()

## 3 · Train and compare three models
Split into training/testing sets. Standardise variables for logistic regression. Performance indicates how well models reconstruct **synthetic labels**, not diagnostic accuracy.

In [ ]:
features = [c for c in data.columns if c != "Simulated_Pattern"]
X, y = data[features], data["Simulated_Pattern"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y)
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}
records = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    records.append({"Model":name,
                    "Accuracy":accuracy_score(y_test, pred),
                    "Precision":precision_score(y_test, pred, zero_division=0),
                    "Recall":recall_score(y_test, pred, zero_division=0),
                    "F1":f1_score(y_test, pred, zero_division=0),
                    "ROC_AUC":roc_auc_score(y_test, prob)})
results = pd.DataFrame(records).sort_values("F1", ascending=False).reset_index(drop=True)
best_name = results.iloc[0]["Model"]
best_model = models[best_name]
print("Selected model:", best_name)
display(results.round(3))
joblib.dump({"model":best_model, "features":features}, "ovarian_synthetic_model.joblib")

## 4 · Confusion matrix and feature importance

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y_test, best_model.predict(X_test)), annot=True, fmt="d",
            cmap="Blues", ax=axes[0], xticklabels=["0","1"], yticklabels=["0","1"])
axes[0].set_title("Test confusion matrix — synthetic outcomes")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
results.set_index("Model")[["Accuracy","F1","ROC_AUC"]].plot.bar(ax=axes[1], ylim=(0,1))
axes[1].set_title("Models compared (holdout test)")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()
print(classification_report(y_test, best_model.predict(X_test), zero_division=0))

## 5 · Build the interactive dashboard
The dashboard contains **Overview**, **Data Explorer**, **Model Performance**, **Pattern Explorer**, **Batch CSV**, and **Project Information**. You can upload a CSV with the same eight feature columns for batch demonstration predictions.

In [ ]:
# The dashboard uses the dataset/model created above.
plot_theme = "plotly_white"

def overview_plots():
    fig1 = px.histogram(data, x="Age", color="Simulated_Pattern", nbins=25,
                        title="Age distribution by simulated label", template=plot_theme)
    fig2 = px.pie(data, names="Simulated_Pattern", title="Synthetic class breakdown",
                  hole=.55, template=plot_theme)
    return fig1, fig2

def explore(feature, pattern):
    view = data if pattern == "All" else data[data.Simulated_Pattern == int(pattern)]
    plot = px.histogram(view, x=feature, nbins=30, color="Simulated_Pattern",
                        title=f"Distribution of {feature}", template=plot_theme)
    return plot, view.head(25), f"{len(view):,} matching demo records"

def performance_plots():
    a = px.bar(results, x="Model", y=["Accuracy", "F1", "ROC_AUC"], barmode="group",
               title="Holdout model comparison", template=plot_theme, range_y=[0,1])
    cm = confusion_matrix(y_test, best_model.predict(X_test))
    b = px.imshow(cm, x=["Predicted 0", "Predicted 1"],
                  y=["Actual 0", "Actual 1"], text_auto=True,
                  title=f"Confusion matrix — {best_name}", color_continuous_scale="Blues")
    return a,b

def predict_demo(age, ca, he4, size, menopause, family, bloating, pain):
    row = pd.DataFrame([[age, ca, he4, size, int(menopause), int(family),
                         int(bloating), int(pain)]], columns=features)
    chance = float(best_model.predict_proba(row)[0,1])
    pattern = "Pattern 1" if chance >= .5 else "Pattern 0"
    message = (f"### {pattern} — {chance:.1%} model score\n"
               "**SIMULATED DATA MODEL ONLY.** This score represents similarity to "
               "fabricated training labels; it is **not** a cancer probability or diagnosis.")
    return message, {"Pattern 0": 1 - chance, "Pattern 1": chance}

def batch_predict(upload):
    if upload is None:
        return pd.DataFrame(), "Upload a CSV with the eight required columns."
    try:
        incoming = pd.read_csv(upload)
        absent = [c for c in features if c not in incoming.columns]
        if absent:
            return pd.DataFrame(), "Missing columns: " + ", ".join(absent)
        if incoming[features].isnull().any().any():
            return pd.DataFrame(), "Missing values found. Complete all feature entries first."
        predicted = best_model.predict_proba(incoming[features])[:,1]
        out = incoming.copy()
        out["Synthetic_Pattern_Score"] = np.round(predicted,3)
        out["Simulated_Prediction"] = np.where(predicted>=.5,1,0)
        out.to_csv("batch_predictions_demo.csv", index=False)
        return out.head(200), f"Processed {len(out):,} records. Results are synthetic-model output only."
    except Exception as err:
        return pd.DataFrame(), f"Could not read the file: {err}"

p1,p2=overview_plots()
m1,m2=performance_plots()
with gr.Blocks(title="Ovarian Health Data • Educational ML Dashboard", theme=gr.themes.Soft()) as dashboard:
    gr.Markdown("# Ovarian Health Data — Machine Learning Dashboard")
    gr.Markdown("**EDUCATIONAL / SYNTHETIC DATA ONLY** · No image analysis · No clinical diagnosis or screening")
    with gr.Tabs():
        with gr.Tab("📊 Overview"):
            gr.Markdown(f"### Dataset: {len(data):,} synthetic records | {len(features)} inputs | 3 models")
            gr.Markdown(f"**Top test F1 model:** {best_name} · **Test F1:** {results.iloc[0]['F1']:.1%} · **Test ROC-AUC:** {results.iloc[0]['ROC_AUC']:.1%}")
            with gr.Row():
                gr.Plot(value=p1)
                gr.Plot(value=p2)
            gr.Dataframe(value=data.head(15), label="Sample rows", interactive=False)
        with gr.Tab("🔎 Data Explorer"):
            with gr.Row():
                feature_choice=gr.Dropdown(features, value="CA125_U_mL", label="Feature")
                pattern_choice=gr.Dropdown(["All","0","1"], value="All", label="Simulated class")
            explore_button=gr.Button("Update exploration")
            explore_chart=gr.Plot(value=explore("CA125_U_mL","All")[0])
            count_text=gr.Markdown(value=f"{len(data):,} matching demo records")
            explore_table=gr.Dataframe(value=data.head(25), interactive=False)
            explore_button.click(explore, inputs=[feature_choice,pattern_choice],
                                 outputs=[explore_chart,explore_table,count_text])
        with gr.Tab("📈 Model Performance"):
            gr.Dataframe(value=results.round(3), interactive=False, label="Model metrics (test set)")
            with gr.Row():
                gr.Plot(value=m1)
                gr.Plot(value=m2)
            gr.Markdown("These performance measures are **not** clinical sensitivity or specificity.")
        with gr.Tab("🧪 Pattern Explorer"):
            gr.Markdown("Enter **fictional sample values** to explore how the synthetic model responds.")
            with gr.Row():
                input_age=gr.Slider(20,80,value=50,step=1,label="Age (years)")
                input_ca=gr.Slider(4,650,value=40,step=1,label="CA-125 (U/mL)")
                input_he4=gr.Slider(12,450,value=75,step=1,label="HE4 (pmol/L)")
                input_size=gr.Slider(.4,12,value=3,step=.1,label="Lesion size (cm)")
            with gr.Row():
                input_meno=gr.Checkbox(label="Postmenopausal")
                input_family=gr.Checkbox(label="Family history")
                input_bloating=gr.Checkbox(label="Persistent bloating")
                input_pain=gr.Checkbox(label="Pelvic pain")
            run_button=gr.Button("Explore simulated pattern", variant="primary")
            out_text=gr.Markdown()
            out_probs=gr.Label(label="Synthetic pattern scores")
            run_button.click(predict_demo,
                inputs=[input_age,input_ca,input_he4,input_size,input_meno,input_family,input_bloating,input_pain],
                outputs=[out_text,out_probs])
        with gr.Tab("📁 Batch CSV"):
            gr.Markdown("Upload a CSV with exactly these eight input fields: `"+"`, `".join(features)+"`. ")
            example=gr.Dataframe(value=data[features].head(5), label="Example input format", interactive=False)
            csv_upload=gr.File(label="Upload CSV", file_types=[".csv"], type="filepath")
            batch_button=gr.Button("Run batch demo")
            batch_table=gr.Dataframe(label="First 200 output rows", interactive=False)
            batch_message=gr.Markdown()
            batch_button.click(batch_predict, inputs=csv_upload, outputs=[batch_table,batch_message])
            gr.Markdown("If you want the complete output CSV after running, download `batch_predictions_demo.csv` from Colab Files.")
        with gr.Tab("ℹ️ About & Limitations"):
            gr.Markdown("""### Project information
- **Project:** Ovarian Cancer Data Analysis — Educational ML Demo
- **Dataset:** 6,000 synthetic, randomly generated rows (no real patients)
- **Inputs:** Age, simulated CA-125, HE4, lesion size, menopause, family history, bloating, pelvic pain
- **Models:** Logistic Regression, Random Forest, Gradient Boosting
- **Evaluation:** 80/20 stratified split; accuracy, precision, recall, F1, ROC-AUC, confusion matrix
- **Limitations:** Artificial relationships and invented labels; neither diagnostic nor screening performance can be inferred.

For real-world ovarian cancer detection, clinically sourced, labelled patient records, appropriate consent, review, external validation and clinician-led governance are necessary.""")

print("Dashboard created successfully. Start it with the final cell.")

## 6 · Launch dashboard in Colab
After running, open the displayed Gradio link. The temporary public link is accessible to anyone who has it, so **do not enter or upload real patient information**. The link expires after the runtime stops.

In [ ]:
# Run this cell LAST to launch the interactive dashboard.
# In Google Colab a temporary share link is printed below.
dashboard.launch(share=True, debug=False)